In [1]:
import os
import chromadb
from langchain_huggingface import HuggingFaceEmbeddings

In [2]:
PERSIST_DIR_HUGGINFACE = "./db_chroma_andino_hugginface"
os.makedirs(PERSIST_DIR_HUGGINFACE, exist_ok=True)

print(f"Carpeta de persistencia (Hugginface): {os.path.abspath(PERSIST_DIR_HUGGINFACE)}")

Carpeta de persistencia (Hugginface): d:\nalvarez\100_cursos\bases_vectoriales\modulos\Chroma_bd\db_chroma_andino_hugginface


In [3]:
client_chroma_hugginface = chromadb.PersistentClient(path=PERSIST_DIR_HUGGINFACE)

collection_hugginface = client_chroma_hugginface.get_or_create_collection(
    name="banco_andino_hugginface"
)

In [4]:

#* Función que generara los embeddings
def embed_hugginface(texts):
    
    embeddings = HuggingFaceEmbeddings(model="ibm-granite/granite-embedding-278m-multilingual")
    response = embeddings.embed_documents(texts)
    
    return response

In [5]:
documentos_banco_andino = [
    {
        "id": "consumo_clasico",
        "texto": """
Crédito de consumo clásico dirigido a personas asalariadas con al menos 12 meses de estabilidad laboral.
Monto: USD 1,000 a 10,000. Plazo: 6 a 36 meses.
Requisitos: comprobante de ingresos, sin mora mayor a 30 días en los últimos 12 meses.
La cuota mensual no debe superar el 35% del ingreso neto del cliente.
""",
    },
    {
        "id": "nomina_convenio",
        "texto": """
Crédito con descuento por nómina para empleados de empresas con convenio vigente con el banco.
No requiere codeudor. Antigüedad mínima: 6 meses.
Monto máximo: hasta 8 veces el salario neto mensual.
El pago se realiza vía deducción automática en planilla.
""",
    },
    {
        "id": "hipotecario_primera_vivienda",
        "texto": """
Crédito hipotecario para primera vivienda.
Financia hasta el 80% del valor de tasación del inmueble. Plazo hasta 20 años.
Requisitos: enganche mínimo del 20%, sin registros negativos en los últimos 24 meses.
Las obligaciones mensuales totales (incluida la hipoteca) no deben superar el 40% del ingreso familiar neto.
""",
    },
    {
        "id": "pyme_capital_trabajo",
        "texto": """
Crédito PYME Capital de Trabajo para empresas con al menos 2 años de operación formal.
Montos desde USD 5,000 hasta USD 200,000. Plazo hasta 24 meses.
Requiere estados financieros, flujo de caja proyectado y, según el riesgo, garantías reales o fideicomisos.
""",
    },
    {
        "id": "politica_riesgo_general",
        "texto": """
Política general de riesgo de crédito.
Se evalúan estabilidad laboral o del negocio, nivel de endeudamiento, score interno y externo,
y comportamiento histórico con el banco.
Solicitudes con endeudamiento total superior al 45% del ingreso neto se consideran solo de forma excepcional.
No se aprueban créditos con moras activas mayores a 90 días al momento de la evaluación.
""",
    },
]

In [6]:
docs_banco_andino = {
    "ids": [doc["id"] for doc in documentos_banco_andino],
    "documents": [doc["texto"] for doc in documentos_banco_andino]
}

In [7]:

#* Generamos una función para mejorar la respuesta de las consultas
def imprimir_resultados(results, titulo="Resultados"):
    print("="*90)
    print(titulo)
    print("="*90)
    
    ids = results.get("ids", [[]])[0]
    documents = results.get("documents", [[]])[0]
    dists = results.get("distances", [[]])[0]
    
    for i, (rid, rdoc, rdist) in enumerate(zip(ids, documents, dists), start=1):
        print(f"Top {i} | ID: {rid:<28} | Distancia: {rdist:.4f}")
        snippet = (rdoc[:180] + '...') if len(rdoc) > 180 else rdoc
        print(f"         | Documento: {snippet}")
        print("-"*90)
    
    print("Nota: menor distancia indica mayor similitud.")

In [8]:

#* Generamos los embeddings con hugginface
embeddings_docs = embed_hugginface(docs_banco_andino['documents'])

print(f"Documentos: {len(docs_banco_andino['documents'])}")
print(f"Embeddings generados: {len(embeddings_docs)}")
print(f"Dimensión de embedding: {len(embeddings_docs[0])}")

Documentos: 5
Embeddings generados: 5
Dimensión de embedding: 768


In [9]:

#* Guardamos los documentos en la colección
collection_hugginface.upsert(
    ids=docs_banco_andino['ids'],
    documents=docs_banco_andino["documents"],
    embeddings=embeddings_docs
)

print("Documentos almacenados en Chroma con embeddings de Hugginface")

Documentos almacenados en Chroma con embeddings de Hugginface


In [10]:
pregunta = "¿Cuál es el monto máximo del crédito con descuento por nómina?"

embedding_pregunta = embed_hugginface([pregunta])[0]

resultado_hugginface = collection_hugginface.query(
    query_embeddings=[embedding_pregunta],
    n_results=3
)

imprimir_resultados(resultado_hugginface)

Resultados
Top 1 | ID: nomina_convenio              | Distancia: 0.3281
         | Documento: 
Crédito con descuento por nómina para empleados de empresas con convenio vigente con el banco.
No requiere codeudor. Antigüedad mínima: 6 meses.
Monto máximo: hasta 8 veces el sal...
------------------------------------------------------------------------------------------
Top 2 | ID: politica_riesgo_general      | Distancia: 0.5527
         | Documento: 
Política general de riesgo de crédito.
Se evalúan estabilidad laboral o del negocio, nivel de endeudamiento, score interno y externo,
y comportamiento histórico con el banco.
Soli...
------------------------------------------------------------------------------------------
Top 3 | ID: consumo_clasico              | Distancia: 0.5772
         | Documento: 
Crédito de consumo clásico dirigido a personas asalariadas con al menos 12 meses de estabilidad laboral.
Monto: USD 1,000 a 10,000. Plazo: 6 a 36 meses.
Requisitos: comprobante de...
------